<a href="https://colab.research.google.com/github/matsunagalab/lecture_ML/blob/main/machine_learning_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 第5回 事後学習: 識別 (分類) その 1 — 基本的事項

この notebook は講義の**事後学習用**です。目安時間は **40分** 程度。

## 到達目標
1. **識別関数** $f(\mathbf{x}|\mathbf{w})$ と **決定境界** $f(\mathbf{x}) = 0$ の関係を説明できる
2. 線形識別を **最小二乗回帰として解く** 素朴な方法を実装できる
3. 最小二乗識別が **異常値に弱い** こと、**ロジスティック回帰**が解決することを理解できる
4. **多クラス分類** を **One-vs-Rest** + One-hot encoding で扱える
5. 識別の **評価指標** — 混同行列 / Precision / Recall / F1 / ROC・AUC — を解釈できる

## 進め方
- 上から順にセルを実行してください。
- 「**回帰のように識別 → 弱点に気づく → ロジスティック回帰で改善 → 多クラスへ拡張 → 性能を測る**」という流れです。


---
# 1. 識別の概念: 識別関数と決定境界

これまで扱ってきた**回帰**は連続値 $y \in \mathbb{R}$ を予測する問題でした。**識別 (classification)** は入力 $\mathbf{x}$ から**離散ラベル** $y \in \{C_0, C_1, \dots\}$ を予測する問題です (文字認識・画像認識・迷惑メール判定など)。

## 識別関数

入力にラベルを当てるルールが**識別関数** $f(\mathbf{x}|\mathbf{w})$。2 クラス ($C_0, C_1$) の場合は

$$
f(\mathbf{x}) > 0 \;\Rightarrow\; \mathbf{x} \in C_1, \qquad f(\mathbf{x}) < 0 \;\Rightarrow\; \mathbf{x} \in C_0
$$

のように**符号でクラスを決める**スタイルが分かりやすいです。

## 決定境界

$f(\mathbf{x}) = 0$ を満たす点の集合が**決定境界 (decision boundary)** で、入力空間をクラスごとの領域に分けます。識別関数を 1 つ決めると決定境界も 1 つ決まる、という対応関係です。


---
# 2. 線形識別

決定境界が**直線・平面・超平面**になる識別を**線形識別**と呼びます。識別関数として線形モデル

$$
f(\mathbf{x}) = \sum_{i=1}^{p} w_i x_i + w_0 = \mathbf{w}^\top \mathbf{x} + w_0
$$

を使うものです (前回までの**線形回帰の延長**として理解できます)。

## 決定境界の幾何

決定境界上の任意の 2 点 $\mathbf{x}_A, \mathbf{x}_B$ は両方 $f = 0$ なので、

$$
\mathbf{w}^\top \mathbf{x}_A + w_0 = 0, \;\; \mathbf{w}^\top \mathbf{x}_B + w_0 = 0
\;\Longrightarrow\;
\mathbf{w}^\top (\mathbf{x}_A - \mathbf{x}_B) = 0
$$

つまり**重みベクトル $\mathbf{w}$ は決定境界に直交**します。$\mathbf{w}$ を求めれば決定境界の向きが決まります。

## 最小二乗法による 2 クラス識別

「識別だから何か特別なアルゴリズムが必要」と身構えず、**まずは回帰として解いて**みます:

- $C_1$ には $y_n = +1$、$C_0$ には $y_n = -1$ とラベルを付ける
- $(\mathbf{x}_n, y_n)$ に最小二乗法で $\mathbf{w}^\top \mathbf{x} + w_0$ を当てはめる
- 予測値の符号でクラスを決める

> **注意**: $y$ が離散値なのに最小二乗を使う根拠は本当はあいまいです。後述の異常値への弱点もそこから来ます。とりあえず動かしてみる、という位置付けです。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
np.random.seed(seed=321)

# 2 次元・2 クラス データ生成 (C_1: y=+1, C_0: y=-1)
N = 50
x_C1 = np.random.randn(N//2, 2) - 1.0;  y_C1 = np.ones((N//2, 1))
x_C0 = np.random.randn(N//2, 2) + 1.0;  y_C0 = -np.ones((N//2, 1))
x = np.concatenate((x_C1, x_C0), axis=0)
y = np.concatenate((y_C1, y_C0), axis=0)

# 切片用の 1 を先頭列に追加して計画行列を作り、最小二乗で w を推定
X = np.hstack((np.ones((N, 1)), x))
clf = LinearRegression(fit_intercept=False).fit(X, y)
print("推定された w:", clf.coef_)

# 識別関数と決定境界 (f=0) を可視化
def f(x1, x2):
    return clf.coef_[0][0] + clf.coef_[0][1]*x1 + clf.coef_[0][2]*x2

x1g = np.linspace(-5, 5, 100); x2g = np.linspace(-5, 5, 100)
X1, X2 = np.meshgrid(x1g, x2g)
plt.contourf(X1, X2, f(X1, X2), levels=[-np.inf, 0, np.inf], alpha=0.3, colors=['white', 'yellow'])
plt.scatter(x_C1[:, 0], x_C1[:, 1], color='blue',  label=r'class $C_1$ (y=+1)')
plt.scatter(x_C0[:, 0], x_C0[:, 1], color='green', label=r'class $C_0$ (y=-1)')
plt.xlim(-5, 5); plt.ylim(-5, 5); plt.xlabel("x1"); plt.ylabel("x2")
plt.legend(loc="upper left"); plt.title("最小二乗識別: 線形決定境界")
plt.show()


**観察**: $f(\mathbf{x}) = 0$ の直線で 2 つのクラスがおおむね分けられています。**回帰の枠組みのまま識別ができる**わけです。


---
# 3. 弱点と解決: 異常値への脆さとロジスティック回帰

## 3.1 最小二乗識別は異常値に弱い

最小二乗法は**全データとの距離の二乗和**を小さくする手法。データが境界から遠く離れていると、その点の二乗誤差が境界を引っ張ってしまいます。「正しい側に十分離れているなら境界に影響しないでほしい」のに、最小二乗ではそうなりません。

## 3.2 ロジスティック回帰: シグモイドで確率に変換

線形モデルの出力 $z = \mathbf{w}^\top \mathbf{x} + w_0$ を**シグモイド関数**

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

に通すと、出力が $(0, 1)$ の範囲に押し込まれ、「クラス $C_1$ に属する**確率**」と解釈できます:

$$
p(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + w_0)
$$

決定境界は $\sigma(f(\mathbf{x})) = 0.5$、すなわち $f(\mathbf{x}) = 0$ で、最小二乗のときと同じく直線・平面です。

損失関数は対数尤度のマイナス:

$$
L(\mathbf{w}, w_0) = - \sum_{n=1}^{N} \left[ y_n \log \sigma(f(\mathbf{x}_n)) + (1 - y_n) \log\bigl(1 - \sigma(f(\mathbf{x}_n))\bigr) \right]
$$

これを**交差エントロピー (cross entropy)** と呼びます。シグモイドは正しい側に十分離れた点で飽和するため、**異常値の寄与がほぼ 0** になり境界が引っ張られません。下で実験で確かめます。


In [ ]:
from sklearn.linear_model import LogisticRegression
np.random.seed(seed=123)

# 60 サンプル: C_1 (20) + C_0 (20) + C_0 異常値 (20)
N = 60
x_C1     = np.random.randn(N//3, 2) - 1.0
x_C0     = np.random.randn(N//3, 2) + 1.0
x_C0_out = np.random.randn(N//3, 2) + [5.0, 10.0]
x = np.concatenate((x_C1, x_C0, x_C0_out), axis=0)
y_pm = np.concatenate((np.ones(N//3), -np.ones(N//3), -np.ones(N//3)))   # 最小二乗用 ±1
y_01 = (y_pm > 0).astype(int)                                             # ロジスティック用 {0,1}

# (a) 最小二乗識別
X_design = np.hstack((np.ones((N, 1)), x))
clf_ls = LinearRegression(fit_intercept=False).fit(X_design, y_pm)
def f_ls(x1, x2): return clf_ls.coef_[0] + clf_ls.coef_[1]*x1 + clf_ls.coef_[2]*x2

# (b) ロジスティック回帰
clf_lr = LogisticRegression().fit(x, y_01)
def f_lr(x1, x2): return clf_lr.intercept_[0] + clf_lr.coef_[0,0]*x1 + clf_lr.coef_[0,1]*x2

# 並べて比較
x1g = np.linspace(-4, 13, 100); x2g = np.linspace(-4, 13, 100)
X1, X2 = np.meshgrid(x1g, x2g)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, ftn, title in [(axes[0], f_ls, "最小二乗識別: 異常値に引っ張られる"),
                       (axes[1], f_lr, "ロジスティック回帰: 異常値に強い")]:
    ax.contourf(X1, X2, ftn(X1, X2), levels=[-np.inf, 0, np.inf], alpha=0.3, colors=['white', 'yellow'])
    ax.scatter(x_C1[:, 0],     x_C1[:, 1],     color='blue',  label=r'$C_1$')
    ax.scatter(x_C0[:, 0],     x_C0[:, 1],     color='green', label=r'$C_0$')
    ax.scatter(x_C0_out[:, 0], x_C0_out[:, 1], color='green', marker='x', s=80, label=r'$C_0$ outlier')
    ax.set_xlim(-4, 13); ax.set_ylim(-4, 13); ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.set_title(title); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()


**観察**: 左 (最小二乗) は異常値に引っ張られて中央クラスタが分けられていません。右 (ロジスティック回帰) は異常値があっても決定境界が中央に収まっています。シグモイドが「正しい側に十分離れた点」の寄与を $\log 1 = 0$ に飽和させるためで、これが両者の決定的な違いです。


---
# 4. 多クラスへの拡張: One-vs-Rest と評価指標

## 4.1 One-hot encoding と One-vs-Rest

クラスが 3 つ以上 ($K \geq 3$) のとき、ラベルを**ワンホット表現** $\mathbf{y} = (0, \dots, 1, \dots, 0)^\top$ で表し、各クラス $k$ について「クラス $k$ vs それ以外」の 2 クラス分類器 $f_k(\mathbf{x})$ を学習。予測時は **argmax** をとります:

$$
\hat{k} = \arg\max_{k \in \{1, \dots, K\}} f_k(\mathbf{x})
$$

決定境界は $f_k(\mathbf{x}) = f_j(\mathbf{x})$ を満たす直線・平面です。

## 4.2 識別の評価指標

回帰では MSE / MAE / $R^2$ で性能を測りましたが、識別では別の指標を使います。

| 指標 | 式 | 解釈 |
|---|---|---|
| Precision | TP / (TP + FP) | P 判定の信頼性 |
| Recall | TP / (TP + FN) | 真の P の取りこぼしのなさ |
| Accuracy | (TP + TN) / 全体 | 全体の正解率。**偏りがあると不適切** |
| F1 | 2 × Precision × Recall / (Precision + Recall) | Precision と Recall の調和平均 |
| AUC | ROC カーブ下の面積 | 閾値非依存。1 に近いほど良い |

> **例**: 1000 人中 1 人だけ陽性の病気を「全員陰性」と答えるモデルは Accuracy 99.9% だが Recall = 0%。Accuracy だけだと有能に見えてしまうので、**偏りのあるデータでは Recall や F1 を併用**します。

## 4.3 Iris データセットでの実験

**Iris (フィッシャーのアヤメ)**: 150 個体、4 特徴量 (がく片 sepal の長さ・幅、花弁 petal の長さ・幅)、3 種 (setosa / versicolor / virginica)。次のセルでは

- (左図) sepal の 2 特徴量で **3 クラス One-vs-Rest** 可視化
- (右図) petal の 2 特徴量で **versicolor vs virginica** の 2 クラス問題に対する **ROC カーブ**

を 1 つの図に並べて実行します。


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             accuracy_score, f1_score, roc_curve, auc)

iris = load_iris()

# (左) 3 クラス One-vs-Rest を sepal の 2 特徴量で可視化
X3 = iris.data[:, :2]
Y = np.zeros((iris.target.size, iris.target.max() + 1))
Y[np.arange(iris.target.size), iris.target] = 1
classifiers = [LinearRegression().fit(X3, Y[:, k]) for k in range(Y.shape[1])]
def predict(x1, x2):
    return np.argmax([c.predict([[x1, x2]])[0] for c in classifiers])

x1g = np.linspace(X3[:, 0].min()-0.5, X3[:, 0].max()+0.5, 80)
x2g = np.linspace(X3[:, 1].min()-0.5, X3[:, 1].max()+0.5, 80)
X1, X2 = np.meshgrid(x1g, x2g)
Z = np.vectorize(predict)(X1, X2)

# (右) versicolor vs virginica を petal で 2 クラス分類し評価指標を計算
mask = iris.target != 0
Xb = iris.data[mask][:, 2:4]                       # 花弁の長さ・幅
yb = (iris.target[mask] == 2).astype(int)          # virginica = 1 (P), versicolor = 0 (N)
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.3, random_state=42, stratify=yb)
clf_b = LogisticRegression().fit(Xtr, ytr)
y_pred  = clf_b.predict(Xte)
y_score = clf_b.predict_proba(Xte)[:, 1]
print("混同行列:\n", confusion_matrix(yte, y_pred))
print(f"Precision: {precision_score(yte, y_pred):.3f}, "
      f"Recall: {recall_score(yte, y_pred):.3f}, "
      f"Accuracy: {accuracy_score(yte, y_pred):.3f}, "
      f"F1: {f1_score(yte, y_pred):.3f}")
fpr, tpr, _ = roc_curve(yte, y_score); roc_auc = auc(fpr, tpr)

# 描画 (3 クラス可視化と ROC を並べる)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].contourf(X1, X2, Z, alpha=0.3, cmap='viridis')
for k, name in enumerate(iris.target_names):
    axes[0].scatter(X3[iris.target == k, 0], X3[iris.target == k, 1], edgecolor='k', label=name)
axes[0].set_xlabel("sepal length"); axes[0].set_ylabel("sepal width")
axes[0].legend(); axes[0].set_title("Iris 3 クラス: 最小二乗 One-vs-Rest")

axes[1].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label="random")
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("versicolor vs virginica の ROC"); axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()


**観察**:
- (左) setosa は線形にきれいに分離できる一方、versicolor と virginica の境界は sepal の 2 特徴量だと曖昧。
- (右) petal の 2 特徴量に切り替えると versicolor と virginica はかなりよく分かれ、AUC は 1 に近い高い値になります。


---
## 演習 5-1

次のうち、**「最小二乗法による線形識別の弱点」とその解決策の組み合わせ** として正しいものをすべて選んでください。

1. 異常値が決定境界を引っ張る → ロジスティック回帰のシグモイド飽和で寄与を抑える
2. 確率を直接出力できない → ロジスティック回帰は $\sigma(f(\mathbf{x}))$ を確率と解釈できる
3. 多クラス分類が原理的に扱えない → One-vs-Rest で扱える
4. 決定境界が必ず非線形になってしまう → 線形モデルなので決定境界は直線・平面


**解答例 (自分で考えてから開いてください)**

**正解: (1), (2)**

- (1) 本文 §3 で実験した通り。
- (2) ロジスティック回帰の出力 $\sigma(f(\mathbf{x}))$ は (0,1) で確率と解釈できる。
- (3) は前半部分が誤り。最小二乗識別でも One-vs-Rest で多クラスを扱える (本文 §4)。
- (4) は前半部分が誤り。最小二乗識別の決定境界は元々**線形** (直線・平面)。

## 演習 5-2

ある病気の検査で 1000 人中 10 人が真陽性 (P)、990 人が真陰性 (N) です。「全員陰性」と答えるモデルの **Accuracy / Precision / Recall** を計算してください。


**解答例 (自分で考えてから開いてください)**

混同行列は

| | 真 P | 真 N |
|---|---|---|
| 予測 P | 0 (TP) | 0 (FP) |
| 予測 N | 10 (FN) | 990 (TN) |

- Accuracy = $(TP+TN)/$ 全体 $= 990/1000 = 0.99$
- Precision = $TP/(TP+FP) = 0/0$ → **未定義**
- Recall = $TP/(TP+FN) = 0/10 = 0$

Accuracy 99% という見かけの良さに騙されないこと。**偏りのあるデータでは Recall や F1 を併用**するのが鉄則です。


---
# まとめ

- **識別** は離散ラベルを予測する問題。識別関数 $f(\mathbf{x}|\mathbf{w})$ の符号でクラスを決め、$f(\mathbf{x}) = 0$ が**決定境界**。
- **線形識別** は $f(\mathbf{x}) = \mathbf{w}^\top \mathbf{x} + w_0$。$\mathbf{w}$ は決定境界に直交。
- **最小二乗識別**: $y = \pm 1$ にして回帰するだけで動くが、**異常値に弱い**。
- **ロジスティック回帰**: シグモイドで確率に変換し**交差エントロピー**を最小化。異常値の影響を受けにくい。
- **多クラス**: One-hot encoding + One-vs-Rest で各クラスごとの 2 クラス分類器を組み合わせる。
- **評価指標**: 混同行列 → Precision / Recall / Accuracy / F1 / ROC・AUC。偏りのあるデータでは Accuracy だけで判断しない。

**次回からの予告**: 識別の発展として **近傍法 (k-NN)** と **サポートベクターマシン (SVM)** に進みます。
